Extract all main campus zip codes and Fall 2023 enrollment data for public and private non-profit 4+ year universities.

In [7]:
import pyodbc
import pandas as pd
from pathlib import Path

# Paths

DB_PATH = Path("data/IPEDS_23-24/IPEDS202324.accdb")
OUT_CSV = Path("data/university_enrollment_zipcode.csv")

# Helpers

def format_ipeds_zip(value):
    """
    IPEDS ZIP fields may be stored without a dash.
    Examples:
      60637      -> 60637
      060102301  -> 06010-2301
    Returns a normalized string or None.
    """
    if pd.isna(value):
        return None

    s = str(value).strip()

    # remove trailing .0 if Access/pandas coerces to float
    if s.endswith(".0"):
        s = s[:-2]

    # keep digits only
    s = "".join(ch for ch in s if ch.isdigit())

    if not s:
        return None

    # 5-digit ZIP
    if len(s) <= 5:
        return s.zfill(5)

    # 9-digit ZIP+4
    if len(s) == 9:
        return f"{s[:5]}-{s[5:]}"

    # anything else: preserve raw digits
    return s


def get_access_connection(db_path: Path):
    conn_str = (
        r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
        f"DBQ={db_path};"
    )
    return pyodbc.connect(conn_str)


# Main

def main():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"Database not found: {DB_PATH}")

    conn = get_access_connection(DB_PATH)

    try:
        # EF2023 has multiple records per institution.
        # EFLEVEL = 10 corresponds to "All students total".
        query = """
        SELECT
            h.UNITID,
            h.INSTNM,
            h.ZIP,
            e.EFTOTAL
        FROM
            HD2023 AS h
            INNER JOIN EF2023 AS e
                ON h.UNITID = e.UNITID
        WHERE
            e.EFLEVEL = 10
            AND h.SECTOR IN (1, 2)
        """

        df = pd.read_sql(query, conn)

    finally:
        conn.close()

    # clean up columns
    df = df.rename(columns={
        "INSTNM": "institution_name",
        "ZIP": "zip_raw",
        "EFTOTAL": "fall_2023_total_enrollment"
    })

    # normalize ZIP
    df["zip_code"] = df["zip_raw"].apply(format_ipeds_zip)

    # optional split fields
    df["zip5"] = df["zip_code"].str.extract(r"^(\d{5})", expand=False)
    df["zip4"] = df["zip_code"].str.extract(r"^\d{5}-(\d{4})$", expand=False)

    # tidy final column order
    df = df[
        [
            "UNITID",
            "institution_name",
            "fall_2023_total_enrollment",
            "zip_code",
            "zip5",
            "zip4",
            "zip_raw",
        ]
    ].sort_values(["institution_name", "UNITID"]).reset_index(drop=True)

    df.to_csv(OUT_CSV, index=False)

    print(f"Saved {len(df):,} rows to: {OUT_CSV.resolve()}")
    print("\nPreview:")
    print(df.head(20).to_string(index=False))


if __name__ == "__main__":
    main()

C:\Users\jwram\AppData\Local\Temp\ipykernel_15912\3780398983.py:81: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Saved 2,427 rows to: C:\Users\jwram\.ipython\university_caffeine_analysis\data\university_enrollment_zipcode.csv

Preview:
 UNITID                                                              institution_name  fall_2023_total_enrollment   zip_code  zip5 zip4    zip_raw
 177834                                       A T Still University of Health Sciences                        3629      63501 63501  NaN      63501
 180203                                                        Aaniiih Nakoda College                         133      59526 59526  NaN      59526
 222178                                                  Abilene Christian University                        5114      79699 79699  NaN      79699
 497037                             Abilene Christian University-Undergraduate Online                         918      75001 75001  NaN      75001
 138558                                          Abraham Baldwin Agricultural College                        3768 31793-2601 31793 2601 31793-

In [ ]:
# pip install pandas geopandas osmnx shapely pyogrio fiona

from pathlib import Path
import re
import time
import pandas as pd
import geopandas as gpd
import osmnx as ox

# Paths

UNIV_CSV = Path("data/university_enrollment_zipcode.csv")
ZCTA_SHP = Path("data/tl_2020_us_zcta520/tl_2020_us_zcta520.shp")
OUT_CSV = Path("caffeine_establishments.csv")
OUT_CSV_AG = Path("caffeine_establishments_overlap.csv")

# harccoded brand whitelist
BRAND_PATTERNS = [
    r"\bstarbucks\b",
    r"\bdunkin\b",
    r"\bdunkin'? donuts\b",
    r"\bpeet'?s\b",
    r"\bcaribou coffee\b",
    r"\btim hortons\b",
    r"\bbiggby\b",
    r"\bthe coffee bean\b",
    r"\bhuman bean\b",
    r"\bscooter'?s coffee\b",
    r"\bphilz\b",
    r"\bblue bottle\b",
    r"\bcoffee bean & tea leaf\b",
]

# tags to ask Overpass/OSM for
# OSMnx interprets values as OR within each key
OSM_TAGS = {
    "amenity": ["cafe"],
    "shop": ["coffee"],
}


# Helpers

def norm_zip(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = "".join(ch for ch in s if ch.isdigit())
    if not s:
        return None
    return s[:5].zfill(5)

def has_explicit_caffeine_signal(row):
    """
    Keep:
      - amenity=cafe
      - shop=coffee
      - explicit coffee-chain brands/names
    Reject:
      - broad fast food unless explicitly whitelisted by brand/name
    """
    vals = []
    for col in ["name", "brand", "brand:wikidata", "amenity", "shop", "cuisine"]:
        if col in row and pd.notna(row[col]):
            vals.append(str(row[col]).lower())
    blob = " | ".join(vals)

    # direct structural includes
    if "amenity" in row and str(row.get("amenity", "")).lower() == "cafe":
        return True
    if "shop" in row and str(row.get("shop", "")).lower() == "coffee":
        return True

    # explicit brand include
    return any(re.search(pat, blob) for pat in BRAND_PATTERNS)


# Load Uni CSV

df = pd.read_csv(UNIV_CSV)

# try a few likely zip column names
zip_col = None
for candidate in ["zip_code", "ZIP", "zip", "zipcode", "zip5"]:
    if candidate in df.columns:
        zip_col = candidate
        break

if zip_col is None:
    raise ValueError("Could not find a ZIP column in the university CSV.")

df["zip5"] = df[zip_col].map(norm_zip)
df = df[df["zip5"].notna()].copy()


# Load ZCTA Polygons

zcta = gpd.read_file(ZCTA_SHP)

# 2020 Census ZCTA shapefiles typically use ZCTA5CE20
if "ZCTA5CE20" not in zcta.columns:
    raise ValueError("Expected ZCTA5CE20 in the ZCTA shapefile.")

zcta["zip5"] = zcta["ZCTA5CE20"].astype(str).str.zfill(5)

needed_zips = sorted(df["zip5"].dropna().unique())
zcta_sub = zcta[zcta["zip5"].isin(needed_zips)].copy()

# project to lat/lon for Overpass
if zcta_sub.crs is None:
    zcta_sub = zcta_sub.set_crs(4269)  # TIGER often comes as NAD83
zcta_sub = zcta_sub.to_crs(4326)


# Query OSM / Overpass per ZIP polygon

ox.settings.timeout = 180
ox.settings.use_cache = True
ox.settings.log_console = True

results = []

for i, row in zcta_sub.iterrows():
    zip5 = row["zip5"]
    geom = row.geometry

    try:
        gdf = ox.features_from_polygon(geom, tags=OSM_TAGS)

        if gdf.empty:
            caffeine_count = 0
        else:
            work = gdf.reset_index(drop=False).copy()

            # flatten columns to plain strings if needed
            work.columns = [str(c) for c in work.columns]

            # keep only explicit caffeine-related places
            mask = work.apply(has_explicit_caffeine_signal, axis=1)
            keep = work[mask].copy()

            # dedupe aggressively: same OSM element only once
            dedupe_cols = [c for c in ["element", "id", "name", "brand", "geometry"] if c in keep.columns]
            if dedupe_cols:
                keep = keep.drop_duplicates(subset=dedupe_cols)
            else:
                keep = keep.drop_duplicates()

            caffeine_count = len(keep)

        results.append({"zip5": zip5, "caffeine_places_zip": caffeine_count})
        print(f"{zip5}: {caffeine_count}")

    except Exception as e:
        print(f"Failed on ZIP {zip5}: {e}")
        results.append({"zip5": zip5, "caffeine_places_zip": pd.NA})

    time.sleep(0.8) # increase or decrease, I need to check the documentation

counts = pd.DataFrame(results)

# Save Outputs

# make sure output parent dirs exist if needed
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_CSV_AG.parent.mkdir(parents=True, exist_ok=True)

# pure ZIP-level counts

zip_counts = counts.copy()

zip_counts = (
    zip_counts
    .dropna(subset=["zip5"])
    .drop_duplicates(subset=["zip5"])
    .sort_values("zip5")
    .reset_index(drop=True)
)

zip_counts.to_csv(OUT_CSV, index=False)

print(f"\nSaved ZIP-level caffeine counts: {OUT_CSV.resolve()}")
print(zip_counts.head(20).to_string(index=False))

# overlap summary when multiple universities share ZIPs

name_col = "institution_name" if "institution_name" in df.columns else "INSTNM"
enroll_col = (
    "fall_2023_total_enrollment"
    if "fall_2023_total_enrollment" in df.columns
    else "EFTOTAL"
)

zip_overlap = (
    df.dropna(subset=["zip5"])
      .groupby("zip5", as_index=False)
      .agg(
          university_count=("UNITID", "nunique"),
          total_enrollment=(enroll_col, "sum"),
          universities=(name_col, lambda s: " | ".join(sorted(set(map(str, s)))))
      )
)

zip_overlap = (
    zip_counts
    .merge(zip_overlap, on="zip5", how="left")
    .sort_values(["university_count", "total_enrollment", "zip5"], ascending=[False, False, True])
    .reset_index(drop=True)
)

zip_overlap.to_csv(OUT_CSV_AG, index=False)

print(f"\nSaved ZIP overlap summary: {OUT_CSV_AG.resolve()}")
print(zip_overlap.head(20).to_string(index=False))

33442: 1
Failed on ZIP 33827: No matching features. Check query location, tags, and log.
